# 1. Descarga de los datos

**Proyecto:** predicción de fuga de clientes (churn) en una empresa de telecomunicaciones.

**Issue:** [#1 Descarga de los datos](https://github.com/velascocafe23/telco-churn-mlops/issues/1)

Este notebook obtiene los datos crudos y documenta el encuadre del problema antes de
cualquier análisis. El dataset no se versiona en el repositorio (`data/` está en el
`.gitignore`), por lo que la descarga se ejecuta desde la fuente original y es
reproducible por cualquier persona que clone el proyecto.

## 1.1 Configuración

In [1]:
from pathlib import Path

import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

RAW_DIR = Path("../../data/01_raw")
RAW_FILE = RAW_DIR / "telco_customer_churn.csv"

FILAS_ESPERADAS = 7043
COLUMNAS_ESPERADAS = 21
OBJETIVO = "Churn"
IDENTIFICADOR = "customerID"

## 1.2 Descarga y almacenamiento de los datos crudos

In [2]:
datos_crudos = pd.read_csv(DATA_URL)

RAW_DIR.mkdir(parents=True, exist_ok=True)
datos_crudos.to_csv(RAW_FILE, index=False)

print(f"Archivo guardado en: {RAW_FILE.resolve()}")
print(f"Tamaño: {RAW_FILE.stat().st_size / 1024:.1f} KB")

Archivo guardado en: /home/velas/proyectos/telco-churn-mlops/data/01_raw/telco_customer_churn.csv
Tamaño: 948.5 KB


## 1.3 Verificación de la descarga

Comprobación mínima de que el archivo descargado corresponde al dataset esperado.
Estas verificaciones son el germen de las reglas de validación que se formalizarán
en la etapa de *data validation* del pipeline de features.

In [3]:
filas, columnas = datos_crudos.shape

print(f"Filas:    {filas}")
print(f"Columnas: {columnas}")
print(f"Memoria:  {datos_crudos.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

assert filas == FILAS_ESPERADAS, f"Se esperaban {FILAS_ESPERADAS} filas"
assert columnas == COLUMNAS_ESPERADAS, f"Se esperaban {COLUMNAS_ESPERADAS} columnas"
assert datos_crudos[IDENTIFICADOR].is_unique, "El identificador de cliente no es único"

Filas:    7043
Columnas: 21
Memoria:  1.86 MB


In [4]:
datos_crudos.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
datos_crudos.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
distribucion_objetivo = datos_crudos[OBJETIVO].value_counts(normalize=True)
print(distribucion_objetivo)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


## 1.4 Encuadre del problema

### ¿Cuál es el objetivo del problema?

Anticipar qué clientes van a cancelar su servicio en el próximo ciclo de facturación,
con suficiente antelación para que el área de retención pueda intervenir. El objetivo
de negocio no es clasificar bien por clasificar: es reducir la tasa de cancelación
actuando sobre los clientes correctos, porque adquirir un cliente nuevo cuesta varias
veces más que retener uno existente.

### ¿Cómo se usará la solución?

Como un proceso batch mensual. El pipeline de inferencia puntúa la base completa de
clientes activos y entrega una lista priorizada por probabilidad de fuga al equipo de
retención, que ejecuta campañas de contacto y ofertas. Adicionalmente, una interfaz de
consulta individual permite evaluar un cliente puntual durante una llamada de servicio.

### ¿Cuáles son las soluciones actuales?

El enfoque tradicional en el sector es la segmentación manual por reglas de negocio:
listas construidas a partir de antigüedad baja, reclamos recientes o vencimiento de
contrato. Son reglas definidas por experiencia, no calibradas, que no entregan una
probabilidad ni permiten priorizar dentro del grupo marcado. Ese es precisamente el
referente contra el que debe competir el modelo.

### ¿Cómo se enmarca el problema?

- **Aprendizaje supervisado**: cada registro tiene etiqueta observada (`Churn`).
- **Clasificación binaria**: el cliente canceló o no canceló.
- **Entrenamiento offline**: el modelo se reentrena periódicamente, no en línea.
- **Inferencia batch**, con un modo de consulta individual como complemento.
- **Datos tabulares estáticos**: un corte transversal, sin componente temporal por
  cliente. Esto último es una limitación relevante y queda registrada como supuesto.

### ¿Cómo se debe medir el desempeño?

Los dos errores no cuestan lo mismo. Un falso negativo es un cliente que se va sin que
nadie lo haya contactado: se pierde todo su valor futuro. Un falso positivo es una
oferta de retención entregada a alguien que se iba a quedar: se pierde el costo de la
oferta, que es mucho menor.

Por esa asimetría la métrica principal es el **recall sobre la clase de fuga**, sujeto a
que la precisión se mantenga en un nivel que haga viable la campaña. Como métrica
secundaria de comparación entre modelos se usa **F1** sobre la clase positiva, y
**ROC-AUC** para evaluar la capacidad de ordenamiento, que es lo que realmente usa el
equipo de retención cuando prioriza una lista.

La exactitud (*accuracy*) se descarta como métrica principal: con una distribución
cercana a 73/27, un clasificador que prediga siempre "no cancela" obtiene alrededor de
0.73 de exactitud sin detectar un solo cliente en riesgo.

### ¿La medida de desempeño está alineada con el objetivo?

Parcialmente, y conviene ser explícito al respecto. El recall mide cuántos clientes en
riesgo se detectan, pero el objetivo real es cuántas cancelaciones se evitan, que además
depende de la efectividad de la campaña de retención. El modelo es una condición
necesaria, no suficiente. La métrica de negocio completa requeriría datos de resultado
de las campañas, que no están en este dataset.

### ¿Cuál es el desempeño mínimo necesario?

El piso lo fija la regla heurística que se formulará en el EDA. El modelo debe superar
de forma estadísticamente significativa ese referente. Como meta operativa inicial se
plantea un recall de al menos 0.70 sobre la clase de fuga manteniendo la precisión por
encima de 0.50, de modo que menos de la mitad del presupuesto de retención se gaste en
clientes que no iban a cancelar.

### ¿Qué problemas parecidos existen y qué se puede reutilizar?

El churn de suscripción es un problema estándar y comparte estructura con el abandono en
banca, seguros y software por suscripción. Son aplicables directamente: el manejo de
desbalance moderado de clases, el uso de modelos de árboles con potenciación del
gradiente como familia de referencia para datos tabulares, y la interpretación por
importancia de atributos para justificar el accionar comercial.

### ¿Cómo se resolvería el problema manualmente?

Un analista con acceso a la base construiría un filtro sobre las variables que la
intuición del negocio señala: contrato mes a mes, antigüedad baja y cargos mensuales
altos. La hipótesis a validar en el EDA es que esa regla simple ya captura una fracción
considerable de las cancelaciones. Esa regla es el modelo base contra el que se compara
todo lo demás.

### ¿Cuál es la fuente de los datos?

Conjunto de muestra de IBM sobre clientes de telecomunicaciones, distribuido
públicamente y ampliamente usado como referencia. Corresponde a una empresa ficticia con
servicios de telefonía e internet.

### ¿Cómo y cada cuánto se actualizan los datos?

La fuente publicada es un archivo estático y no se actualiza. En un entorno productivo
real, el equivalente sería una extracción del sistema de facturación con periodicidad
mensual, alineada al ciclo de facturación que define la etiqueta de cancelación. Esa
cadencia mensual es la que determina la frecuencia del pipeline de inferencia y la
ventana razonable de reentrenamiento.

## 1.5 Supuestos

1. La etiqueta `Churn` corresponde a cancelación en el último mes, coherente con la
   documentación del conjunto de datos.
2. Los registros son independientes entre sí: un cliente aparece una sola vez.
3. No hay componente temporal explotable. Se trabaja con un corte transversal, de modo
   que no aplican particiones temporales ni análisis de secuencia.
4. La distribución del conjunto es representativa de la población de clientes. No hay
   forma de verificarlo con la información disponible.
5. Todas las variables presentes están disponibles en el momento de la predicción. Esto
   debe revisarse en el EDA: si alguna se registra después de la cancelación,
   introduciría fuga de información.
6. El costo de un falso negativo es sustancialmente mayor que el de un falso positivo,
   aunque no se dispone de las cifras para cuantificar la relación exacta.
7. `customerID` es un identificador sin valor predictivo y se descartará en el
   preprocesamiento.

## 1.6 Observaciones para la siguiente etapa

La salida de `info()` muestra algo que hay que atender de inmediato: **`TotalCharges` fue
leída como texto**, pese a ser un valor monetario acumulado. Es el primer problema de
calidad detectado y se resuelve en el notebook de exploración inicial, junto con la
unificación de la representación de los valores nulos y la conversión de tipos.

El objetivo presenta desbalance moderado, lo que condiciona tres decisiones posteriores:
la estratificación en la partición train/test, la elección de métricas, y el tratamiento
del desbalance durante el entrenamiento.